# Bathymetry — depth distribution

Pull a GEBCO 2020 subset of a deep Atlantic abyssal plain and summarise its
**depth distribution**: a histogram of cell depths plus the cumulative
(hypsometric) curve. This is the quick statistical read a user makes to
characterise a basin — modal depth, spread, deepest point.

In [ ]:
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from pyramids.dataset import Dataset

from earthlens.core import EarthLens

In [ ]:
out = tempfile.mkdtemp()
paths = EarthLens(
    data_source='bathymetry',
    dataset='gebco_2020',
    aoi=[-40.0, 20.0, -38.0, 22.0],
    path=out,
).download(progress_bar=False)

arr = np.asarray(Dataset.read_file(paths[0]).read_array(), dtype='float32')
arr = arr[0] if arr.ndim == 3 else arr
depths = arr[np.isfinite(arr) & (arr != 32767)]
print('cells', depths.size)
print('median depth', round(float(np.median(depths))), 'm')
print(
    'deepest',
    round(float(depths.min())),
    'm | shallowest',
    round(float(depths.max())),
    'm',
)

## Histogram + hypsometric curve

The histogram shows how much sea floor sits at each depth band; the cumulative
curve (right axis) reads off the fraction of the area shallower than a depth.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(depths, bins=40, color='#3182bd')
ax.set_xlabel('elevation (m)')
ax.set_ylabel('cell count', color='#3182bd')
ax.axvline(np.median(depths), color='crimson', ls='--', lw=1, label='median')

order = np.sort(depths)
cum = np.linspace(0, 100, order.size)
ax2 = ax.twinx()
ax2.plot(order, cum, color='k', lw=1.2)
ax2.set_ylabel('cumulative area (%)')
ax.set_title('GEBCO 2020 depth distribution — central Atlantic abyssal plain')
ax.legend(loc='upper left')
plt.show()